# Meeting cost calculator

Real people, real pay, from [SF Employee Compensation](https://data.sfgov.org/City-Management-and-Ethics/Employee-Compensation/88g8-5mnd/about_data).
Each person's hourly rate is *their own* total compensation divided by *their own* hours worked.

In [2]:
import pandas as pd, requests

rows, offset = [], 0
while True:
    r = requests.get("https://data.sfgov.org/resource/88g8-5mnd.json", params={
        "$select": "employee_identifier as name, job, department, hours, total_compensation",
        "$where": "year_type='Fiscal' AND year='2026' AND hours > 0",
        "$limit": 50000, "$offset": offset}, timeout=180)
    r.raise_for_status()
    batch = r.json()
    rows += batch
    if len(batch) < 50000: break
    offset += 50000

emp = pd.DataFrame(rows)
emp[["hours", "total_compensation"]] = emp[["hours", "total_compensation"]].apply(pd.to_numeric)
emp["rate"] = emp["total_compensation"] / emp["hours"]
print(f"{len(emp):,} employees, FY2026")

42,456 employees, FY2026


## Look someone up

Names repeat (there are eight Kevin Lees), so check before you trust a match.

In [10]:
def find(name):
    return emp[emp["name"].str.contains(name, case=False)][
        ["name", "job", "department", "hours", "total_compensation", "rate"]]

find("vang")

,name,job,department,hours,total_compensation,rate
6574,Danny Vang,Administrative Analyst,City Administrator,2160.00,174917.59,80.980366
7379,Thomas Vang,Special Nurse,Public Health,652.00,111189.96,170.536748
7916,Pakau Vang,Sr Payroll & Personnel Clerk,Controller,2216.00,153406.97,69.226972
11696,Daisy Evangelista,Registered Nurse,Public Health,2222.58,354367.90,159.439885
13276,Evangeline Lim,Senior Administrative Analyst,Municipal Transportation Agcy,2160.00,208949.27,96.735773
14913,Evangelina Tongol,Medical Assistant,Public Health,761.43,56733.51,74.509160
16503,Roselle Evangelista,Registered Nurse,Public Health,2161.00,311797.14,144.283730
17267,Surai Vang,Pharmacy Technician,Public Health,2465.73,207952.92,84.337263
19003,Evangeline Velasquez,Tech Analyst/Designer-Senior,Municipal Transportation Agcy,2160.00,227528.35,105.337199
19295,Susan Vang-Chan,Senior Human Resources Analyst,Airport Commission,1760.42,170650.40,96.937322


## Edit this cell and run it

List each attendee by name. If a name is ambiguous, add a department hint:
`("Kevin Lee", "Police")`.

In [11]:
ATTENDEES = [
    "Katharine Petrucione",
    "Michael Makstman",
    "Nathan Sinclair",
    "Sophia Kittler",
    "Rafael Mandelman",
    "Angela Calvillo",
    "Greg Wagner",
    "Carol Isen",
    "Daniel Tsai",
    "Dennis Herrera",
    "Michael Lambert",
    "Mary Ellen Carroll",
    "Naiyapakorn Nakornkhet",
    "Julie Kirschbaum",
    "Trent Rhorer",
    "Mawuli Tugbenyoh",
    "Edward McCaffrey",
    "Julia Chrusciel",
    "Damon Daniels",
    "Danny Vang"
    
]
MINUTES = 120

hours, total = MINUTES / 60, 0
for a in ATTENDEES:
    name, dept = a if isinstance(a, tuple) else (a, "")
    hit = emp[emp["name"].str.lower() == name.lower()]
    if dept:
        hit = hit[hit["department"].str.contains(dept, case=False)]

    if len(hit) == 0:
        print(f"{name:22} NOT FOUND - try find({name!r})")
        continue
    if len(hit) > 1:
        print(f"{name:22} AMBIGUOUS - {len(hit)} matches, add a dept hint:")
        print(hit[["job", "department", "rate"]].to_string(index=False))
        continue

    p = hit.iloc[0]
    cost = p["rate"] * hours
    total += cost
    print(f"{name:22} {p['job'][:28]:28} ${p['rate']:6,.0f}/hr   ${cost:8,.2f}")

print(f"\n{MINUTES} minutes x {len(ATTENDEES)} people = ${total:,.2f}")

Katharine Petrucione   Dep Dir IV                   $   199/hr   $  398.34
Michael Makstman       Dept Head IV                 $   205/hr   $  409.60
Nathan Sinclair        Dep Dir IV                   $   175/hr   $  349.07
Sophia Kittler         Mayoral Staff XVII           $   140/hr   $  280.41
Rafael Mandelman       Member, Board of Supervisors $   115/hr   $  229.90
Angela Calvillo        Dept Head III                $   175/hr   $  350.75
Greg Wagner            NOT FOUND - try find('Greg Wagner')
Carol Isen             Human Resources Director     $   214/hr   $  428.23
Daniel Tsai            Dept Head V                  $   289/hr   $  577.35
Dennis Herrera         Executive Contract Employee  $   287/hr   $  573.41
Michael Lambert        Dept Head IV                 $   207/hr   $  414.31
Mary Ellen Carroll     Dept Head IV                 $   213/hr   $  426.44
Naiyapakorn Nakornkhet Dept Head V                  $   242/hr   $  484.39
Julie Kirschbaum       Gen Mgr, Public Tr

Rate is total compensation (salary + overtime + benefits) over hours actually worked,
so it's what the city pays for an hour of that person's time.